In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
import pandas as pd

In [3]:
df  = pd.read_csv('../E-commerce_data/Cleaned.csv')

In [4]:
df.head(2)

,Unnamed: 0.1,Unnamed: 0,OrderId,CustomerId,OrderDate,Age,City,Category,ProductName,Quantity,...,ShippingDays,DeliveryDistanceKM,Device,MarketingChannel,Spend90d,CouponCode,TotalAmount,Year,Month,Day
0,0,0,ORD1048298,CUST55152,NaN,33,Lucknow,Grocery,Bluetooth Speaker,1.0,...,4.0,4.74,Desktop,Social,10552.73,NEWUSER,1076.16,NaN,NaN,NaN
1,1,1,ORD1081047,CUST35058,2025-08-14,28,Chennai,Home & Kitchen,Running Shoes,1.0,...,3.0,16.72,Mobile,Social,894.36,FESTIVE,1015.11,2025.0,8.0,14.0


In [5]:
df.drop(columns=['Unnamed: 0.1','Unnamed: 0'],inplace=True)

In [7]:
df.to_csv('Cleaned_data.csv')

In [ ]:
data = df.copy()

data.shape

(88258, 25)

In [ ]:
target = "TotalAmount"

X = data.drop(columns=[target])
y = data[target]

In [ ]:
X.shape, y.shape

((88258, 24), (88258,))

In [ ]:
X["OrderDate"] = pd.to_datetime(
    X["OrderDate"],
    errors="coerce"
)

In [ ]:
X["OrderYear"] = X["OrderDate"].dt.year
X["OrderMonth"] = X["OrderDate"].dt.month
X["OrderDay"] = X["OrderDate"].dt.day
X["OrderDayOfWeek"] = X["OrderDate"].dt.dayofweek

In [ ]:
X.drop(columns=["OrderDate"], inplace=True)

In [ ]:
id_columns = [
    "Unnamed: 0",
    "Unnamed: 0.1",
    "OrderId",
    "CustomerId"
]

X.drop(
    columns=id_columns,
    errors="ignore",
    inplace=True
)

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

In [ ]:
print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Age', 'Quantity', 'UnitPrice', 'DiscountPct', 'Rating', 'ShippingDays', 'DeliveryDistanceKM', 'Spend90d', 'Year', 'Month', 'Day', 'OrderYear', 'OrderMonth', 'OrderDay', 'OrderDayOfWeek']

Categorical Features:
['City', 'Category', 'ProductName', 'PaymentMethod', 'OrderStatus', 'Device', 'MarketingChannel', 'CouponCode']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (70606, 23)
X_test : (17652, 23)
y_train: (70606,)
y_test : (17652,)


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

In [ ]:
models = {

    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(),

    "Lasso": Lasso(),

    "ElasticNet": ElasticNet(),

    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        random_state=42
    )
}

In [ ]:
param_grids = {

    "Linear Regression": {},

    "Ridge": {
        "model__alpha": [
            0.01,
            0.1,
            1,
            10,
            100
        ]
    },

    "Lasso": {
        "model__alpha": [
            0.0001,
            0.001,
            0.01,
            0.1,
            1
        ]
    },

    "ElasticNet": {
        "model__alpha": [
            0.0001,
            0.001,
            0.01,
            0.1,
            1
        ],
        "model__l1_ratio": [
            0.1,
            0.3,
            0.5,
            0.7,
            0.9
        ]
    },

    "Random Forest": {
        "model__n_estimators": [
            100,
            200,
            300
        ],
        "model__max_depth": [
            None,
            10,
            20,
            30
        ],
        "model__min_samples_split": [
            2,
            5,
            10
        ],
        "model__min_samples_leaf": [
            1,
            2,
            4
        ]
    },

    "Extra Trees": {
        "model__n_estimators": [
            100,
            200,
            300
        ],
        "model__max_depth": [
            None,
            10,
            20,
            30
        ],
        "model__min_samples_split": [
            2,
            5,
            10
        ]
    },

    "Gradient Boosting": {
        "model__n_estimators": [
            100,
            200
        ],
        "model__learning_rate": [
            0.01,
            0.05,
            0.1
        ],
        "model__max_depth": [
            2,
            3,
            5
        ]
    },

    "AdaBoost": {
        "model__n_estimators": [
            50,
            100,
            200
        ],
        "model__learning_rate": [
            0.01,
            0.05,
            0.1,
            0.5,
            1
        ]
    }
}

In [ ]:
results = []

best_models = {}

In [ ]:
# for model_name, model in models.items():

#     print("=" * 70)
#     print(f"Training: {model_name}")

#     pipeline = Pipeline(
#         steps=[
#             ("preprocessor", preprocessor),
#             ("model", model)
#         ]
#     )

#     param_grid = param_grids[model_name]

#     if param_grid:

#         search = RandomizedSearchCV(
#             estimator=pipeline,
#             param_distributions=param_grid,
#             n_iter=10,
#             cv=5,
#             scoring="neg_root_mean_squared_error",
#             random_state=42,
#             n_jobs=-1
#         )

#         search.fit(X_train, y_train)

#         best_model = search.best_estimator_

#         print("Best Parameters:")
#         print(search.best_params_)

#     else:

#         # pipeline.fit(
#             X_train,
#             y_train
#         )

#         best_model = pipeline

#     best_models[model_name] = best_model

#     y_pred = best_model.predict(X_test)

#     mae = mean_absolute_error(
#         y_test,
#         y_pred
#     )

#     mse = mean_squared_error(
#         y_test,
#         y_pred
#     )

#     rmse = np.sqrt(mse)

#     r2 = r2_score(
#         y_test,
#         y_pred
#     )

#     results.append({

#         "Model": model_name,

#         "MAE": mae,

#         "MSE": mse,

#         "RMSE": rmse,

#         "R2_Score": r2

#     })

#     print(f"MAE  : {mae:.4f}")
#     print(f"MSE  : {mse:.4f}")
#     print(f"RMSE : {rmse:.4f}")
#     print(f"R2   : {r2:.4f}")

Training: Linear Regression
MAE  : 24181.1874
MSE  : 44572142938.4281
RMSE : 211121.1570
R2   : -0.0012
Training: Ridge


c:\Users\md salman\anaconda3\Anaconda\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Parameters:
{'model__alpha': 100}
MAE  : 24150.8609
MSE  : 44570879128.5317
RMSE : 211118.1639
R2   : -0.0012
Training: Lasso


c:\Users\md salman\anaconda3\Anaconda\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Parameters:
{'model__alpha': 1}
MAE  : 24175.2295
MSE  : 44571897095.3957
RMSE : 211120.5748
R2   : -0.0012
Training: ElasticNet
Best Parameters:
{'model__l1_ratio': 0.7, 'model__alpha': 1}
MAE  : 23844.5212
MSE  : 44533349181.1710
RMSE : 211029.2614
R2   : -0.0003
Training: Random Forest


In [ ]:
results_df = pd.DataFrame(results)

results_df

In [ ]:
print("Target null values:", y.isnull().sum())

Target null values: 785


In [ ]:
valid_target = y.notna()

X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Target NaN:", y.isnull().sum())

X shape: (87473, 23)
y shape: (87473,)
Target NaN: 0


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
print("y_train NaN:", y_train.isnull().sum())
print("y_test NaN:", y_test.isnull().sum())

y_train NaN: 0
y_test NaN: 0
